In [ ]:
import numpy as np
import pandas as pd
from gbmcollaboration import AllDataCenters
from gbmcollaboration import (
    SURVIVAL_EXCLUSION
)
from gbmcollaboration.viz.cell2loc_plotting import clustermap
from gbmcollaboration.viz.figure_helpers import savefig, set_mpl_rcparams
# from gbmcollaboration.utils import get_datasets, s3_ls
# data = get_datasets()
set_mpl_rcparams()

In [ ]:
# s3_bucket="pa-bi4jtrdsa151p4d3m-jepjxh9qrwgxphe8hd3r6exicahnoeuw1b-s3alias"
# s3_prefixes=(
#     "fd319842-9157-4f04-bd4f-7a36ce6a0ba7/Non_mosaic_upload/Mosaic Annotations/",  # CHUV
#     "2ca49430-8b5c-4504-ad96-344019c90e84/",  # UKER
# )

# Params

In [ ]:
path_efs = '/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/'
path_ecm_annot = path_efs+'Analysis_Repo/Notebooks/ecm_neighborhood_annotations_v4.csv'
path_til_annot = path_efs+'results/visium/til_zones/til_zone_annotations.csv'
path_cell_type_info = '../../gbmcollaboration/metadata/Cell type annotation markers Glimmer.xlsx'

# Load data

In [ ]:
# data = AllDataCenters.load_spatial(
#     exclude_low_quality_visium = True,
#     exclude_idh1_mut = True,
#     exclude_patient_ids = True,
#     exclude_relapse = True,
#     load_deconvolution=True,
#     load_gex=False
# )

df_q05 = pd.concat([v.celltype_df.drop(columns=['imagerow','imagecol']) for k,v in data.items()]).fillna(0.)
df_q05.index = df_q05.index.to_flat_index().str.join('_')

# map cell types names to paper names
df_markers, df_ct_map = pd.read_excel(path_cell_type_info,sheet_name=None).values()
_map = df_ct_map.set_index('celltype_level4_scanvi')[f'celltype_level4_curated']

assert all(df_q05.columns.isin(_map.index))
df_q05.columns = df_q05.columns.map(_map)
ct_cols = df_q05.columns.tolist()

# normalized version
df_q05_norm = df_q05.div(df_q05.sum(axis=1), axis=0)

# join with metadata
df_ecm_annot = pd.read_csv(path_ecm_annot,index_col='spot_id')
df_til_annot = pd.read_csv(path_til_annot,index_col='spot_id')

assert all(df_q05.index == df_ecm_annot.index)
assert all(df_q05.index == df_til_annot.index)

df_q05 = df_q05.join([df_ecm_annot,df_til_annot.drop(columns='sample')])
df_q05_norm = df_q05_norm.join([df_ecm_annot,df_til_annot.drop(columns='sample')])

In [ ]:
df_til_annot

# Claims
- We observed colocalization of astrocytes and neuronal subtypes in samples with adjacent tumor-infiltrated brain (Fig. 4a) and hypoxia-associated cell types including MES1, MES_hypoxia, LAMs, and TREM2-positive microglia (Fig. 4d) 

- Additionally we identified interferon responsive T cells and MDMs in overlapping niches (Fig. 4c). 

- We also observed stratified patterns of tumor cell subtypes transitioning from hypoxia-associated cores to OPC-like and NPC-like interfaces with infiltrated brain (Fig. 4b).

- We assessed the spatial distribution of ECM neighborhoods to evaluate alignment with known tumor topography (Fig. 4h). ECM neighborhood 4 was primarily associated with tumor-infiltrated brain tissue adjacent to the tumor mass, defined by comparative depletion of ECM expression across all panel genes (logFC -2.9, Supp. Table 4x) and high prevalence of neuronal populations (Fig. 4g)
    - Elo's comment: On the stacked barplot of 12 mot abundant cell types the distinguishing cell types are: Macrophage_LAM and neuron_excitatory_nos are in this one and each are only in one other cluster. Neutrophils and plasma cells are exclusively in that cluster.


In [ ]:
plot_annot = 'TIL_zone'

if plot_annot == 'ECM_neighborhood':
    col_prefix = 'ECM_'
elif plot_annot == 'TIL_zone':
    col_prefix = ''
df_q05_mean = df_q05.groupby(plot_annot)[ct_cols].mean(0).T
df_q05_norm_mean = df_q05_norm.groupby(plot_annot)[ct_cols].mean(0).T

In [ ]:
upper=np.quantile(df_q05_norm_mean,0.99)
df_plot = df_q05_norm_mean.copy()
df_plot = df_plot.clip(upper=upper)
df_plot.columns = col_prefix+df_plot.columns.astype(str)

# ct_subset_ecm = ['Macrophage_LAM','Neuron_excitatory_nos','Neutrophil','Plasma_cell']
# df_plot_subset = df_plot.loc[ct_subset_ecm]

clustermap(df_plot, figure_size=(14, 14), 
            cmap='RdPu', log=True, fun_type='dotplot',
            cluster_rows=True, cluster_cols=True)

# clustermap(df_plot_subset, figure_size=(5, 5), 
#             cmap='RdPu', log=True, fun_type='dotplot',
#             cluster_rows=True, cluster_cols=True)

In [ ]:
for a in df_q05[plot_annot].unique():
    ind = adata_vis.obs[a] == '1.0'
    df_q05[a] = adata_vis.obs.loc[ind, adata_vis.uns['mod']['factor_names']].mean(0)
    
annot_abundance_norm = (df_q05.T / df_q05.sum(1)).T

clustermap(annot_abundance_norm, figure_size=(5, 11), 
            cmap='RdPu', log=True, fun_type='dotplot',
            cluster_rows=True, cluster_cols=True)

In [ ]:
ad_.obsm['means_cell_abundance_w_sf']